# 07 — Betting Integration
## Odds Processing, Edge Detection, Kelly Sizing

**Purpose:** Connect model probabilities to real sportsbook odds and identify profitable bets.

**Pipeline:**
1. **Odds Processing** — Remove bookmaker overround (Shin's method)
2. **Edge Detection** — Compare P_model vs P_market
3. **Kelly Sizing** — Determine optimal stake per bet
4. **Bankroll Management** — Track P&L and enforce exposure limits


In [ ]:
# --- Setup ---
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from config.settings import Settings
from betting.odds_processing import (
    decimal_odds_to_implied_prob,
    compute_overround,
    remove_overround_shin,
    remove_overround_proportional,
    remove_overround_power,
    process_tournament_odds,
)
from betting.edge_detection import EdgeDetector
from betting.kelly import KellyCalculator, kelly_fraction
from betting.bankroll import BankrollTracker

cfg = Settings()


In [ ]:
# --- Overround Removal Demo ---
# Synthetic odds for a 10-player field
np.random.seed(42)
true_probs = np.array([0.15, 0.12, 0.10, 0.10, 0.08, 0.08, 0.07, 0.07, 0.06, 0.17])
true_probs /= true_probs.sum()

# Add 25% overround (typical for golf outrights)
overround = 0.25
raw_implied = true_probs * (1 + overround)
decimal_odds = 1.0 / raw_implied

print("=== OVERROUND REMOVAL COMPARISON ===")
print(f"Raw implied sum: {raw_implied.sum():.3f} (overround: {compute_overround(raw_implied)*100:.1f}%)")
print()

shin_probs, z = remove_overround_shin(raw_implied)
prop_probs = remove_overround_proportional(raw_implied)
power_probs = remove_overround_power(raw_implied)

comparison = pd.DataFrame({
    "True": np.round(true_probs, 4),
    "Raw Implied": np.round(raw_implied, 4),
    "Shin": np.round(shin_probs, 4),
    "Proportional": np.round(prop_probs, 4),
    "Power": np.round(power_probs, 4),
})
comparison["Shin Error"] = np.round(np.abs(shin_probs - true_probs), 5)
comparison["Prop Error"] = np.round(np.abs(prop_probs - true_probs), 5)
print(comparison.to_string(index=False))

print(f"\nShin insider fraction z = {z:.4f}")
print(f"Shin total error:         {np.sum(np.abs(shin_probs - true_probs)):.5f}")
print(f"Proportional total error: {np.sum(np.abs(prop_probs - true_probs)):.5f}")
print(f"Power total error:        {np.sum(np.abs(power_probs - true_probs)):.5f}")


In [ ]:
# --- Kelly Criterion Demo ---
print("=== KELLY CRITERION ===\n")

test_cases = [
    (0.05, 25.0, "Moderate edge on longshot"),
    (0.10, 12.0, "Strong edge on mid-range"),
    (0.03, 40.0, "Small edge on extreme longshot"),
    (0.20, 6.0,  "Very strong favorite edge"),
]

for p, odds, desc in test_cases:
    full_k = kelly_fraction(p, odds)
    frac_k = full_k * cfg.KELLY_FRACTION
    ev = p * odds - 1  # Expected value per unit
    
    print(f"{desc}:")
    print(f"  P_model={p:.1%}, Odds={odds:.1f}, EV={ev:+.2f}")
    print(f"  Full Kelly:  {full_k:.4%}")
    print(f"  {cfg.KELLY_FRACTION:.0%} Kelly: {frac_k:.4%} → ${frac_k * cfg.INITIAL_BANKROLL:.2f}")
    print()


In [ ]:
# --- Full Pipeline Demo: Edge Detection + Sizing ---
# Simulate model predictions and market odds for a tournament

model_probs = {
    1001: 0.08, 1002: 0.06, 1003: 0.05, 1004: 0.04, 1005: 0.03,
    1006: 0.03, 1007: 0.02, 1008: 0.02, 1009: 0.015, 1010: 0.01,
}

market_df = pd.DataFrame({
    "player_id": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010],
    "true_prob":  [0.07, 0.05, 0.06, 0.035, 0.025, 0.04, 0.015, 0.025, 0.01, 0.012],
    "decimal_odds": [14.3, 20.0, 16.7, 28.6, 40.0, 25.0, 66.7, 40.0, 100.0, 83.3],
    "book": ["pinnacle"] * 10,
})

# Detect edges
detector = EdgeDetector(cfg)
opportunities = detector.find_edges(model_probs, market_df)
print(f"Edges found: {len(opportunities)}")
print(detector.opportunities_to_dataframe(opportunities).to_string(index=False))

# Size bets
kelly_calc = KellyCalculator(cfg, bankroll=cfg.INITIAL_BANKROLL)
bets = kelly_calc.size_bets(opportunities)
print(f"\nSized bets: {len(bets)}")
print(kelly_calc.bets_to_dataframe(bets).to_string(index=False))


In [ ]:
# --- H2H Matchup Edge Detection Demo ---
# For matchups, the market is 2-way: Player A vs Player B.
# Overround is simpler (two outcomes, not 150+).

print("=== H2H MATCHUP EDGE DETECTION ===\n")

# Simulated example: model thinks Player A has 60% chance to beat Player B
# Book offers odds: A at 1.85, B at 2.05

model_p_a = 0.60
model_p_b = 0.40

odds_a = 1.85  # decimal
odds_b = 2.05

# Devig: proportional removal
raw_p_a = 1.0 / odds_a
raw_p_b = 1.0 / odds_b
overround = raw_p_a + raw_p_b
market_p_a = raw_p_a / overround
market_p_b = raw_p_b / overround

print(f"Book odds: A={odds_a:.2f}, B={odds_b:.2f}")
print(f"Raw implied: A={raw_p_a:.3f}, B={raw_p_b:.3f} (sum={overround:.3f}, vig={overround-1:.1%})")
print(f"Devigged:    A={market_p_a:.3f}, B={market_p_b:.3f}")
print(f"Model:       A={model_p_a:.3f}, B={model_p_b:.3f}")
print()

# Edge calculation
edge_a = model_p_a - market_p_a
edge_b = model_p_b - market_p_b
print(f"Edge on A: {edge_a:.3f} ({edge_a*100:.1f}%)")
print(f"Edge on B: {edge_b:.3f} ({edge_b*100:.1f}%)")

# Kelly sizing for the side with edge
if edge_a > 0.05:
    b = odds_a - 1
    q = 1 - model_p_a
    full_k = (model_p_a * b - q) / b
    frac_k = full_k * 0.25
    bankroll = 5000
    stake = frac_k * bankroll
    ev_per_bet = model_p_a * b - q
    print(f"\nBet on A: Kelly f*={full_k:.4f}, 25% Kelly={frac_k:.4f}")
    print(f"  Stake: ${stake:.2f} (of ${bankroll:,} bankroll)")
    print(f"  EV per $1: ${ev_per_bet:.3f}")

print("\n--- Key difference from outright betting ---")
print("  Outright: 150+ outcomes, Shin devig, P(win) ~ 1-10%")
print("  Matchup:  2 outcomes, proportional devig, P(win) ~ 40-60%")
print("  Result:   More bets per event, smaller edges, but much larger sample size")

---
## ✅ Betting Integration Complete

**Key takeaways:**
- Shin's method is most accurate for removing overround in golf markets
- Fractional Kelly (25%) provides safety against estimation error
- Tournament exposure caps prevent over-concentration

**Next step:** → **08_full_backtest.ipynb** (the real test)
